In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ---------------------------------------------------------
# 1. DATA LOADING AND PRE-PROCESSING
# ---------------------------------------------------------

# Load the csv
filename = "virus_spread_batch_with_firstpeak_indicator.csv"
modelDat = pd.read_csv(filename)

# Rename columns as requested
# Mapping 'iteration' to 'Seed'.
modelDat = modelDat.rename(columns={"Step": "Day", "iteration": "Seed"})

# Identify the Strategy names based on columns ending with '_InfectionPercent'
strategy_cols = [c for c in modelDat.columns if "_InfectionPercent" in c]
strategies = [s.replace("_InfectionPercent", "") for s in strategy_cols]

print(f"Strategies detected: {strategies}")

# ---------------------------------------------------------
# 2. EXTRACT PEAK DATA (New DataFrame creation)
# ---------------------------------------------------------

peak_records = []

# We iterate through the dataset to find rows where IsFirstPeak == 1
for strategy in strategies:
    peak_col = f"{strategy}_IsFirstPeak"
    inf_col = f"{strategy}_InfectionPercent"
    
    # Filter for rows where this strategy hit its first peak
    peaks = modelDat[modelDat[peak_col] == 1].copy()
    
    if not peaks.empty:
        peaks['Strategy'] = strategy
        peaks['Peak_Infection_Rate'] = peaks[inf_col]
        
        # Select relevant columns including seed_country and param_travel_rate
        subset = peaks[[
            'Seed', 'Day', 'seed_country', 'param_travel_rate', 
            'Strategy', 'Peak_Infection_Rate'
        ]]
        
        peak_records.append(subset)

# Combine all strategies into one dataframe
if peak_records:
    peaks_summary = pd.concat(peak_records)
    # Sort for consistency
    peaks_summary = peaks_summary.sort_values(
        by=['param_travel_rate', 'seed_country', 'Seed', 'Strategy', 'Day']
    )
    print("Peak summary extracted.")
else:
    print("No peaks found in the dataset.")
    peaks_summary = pd.DataFrame() 

# Helper function to add labels to bar charts
def add_bar_labels(ax, fmt='%.1f'):
    for container in ax.containers:
        ax.bar_label(container, fmt=fmt, padding=3)

# ---------------------------------------------------------
# 3. MAIN LOOP: TRAVEL RATE -> SEED COUNTRY
# ---------------------------------------------------------

# Get all unique Travel Rates and Seed Countries
unique_rates = sorted(modelDat['param_travel_rate'].unique())
unique_countries = sorted(modelDat['seed_country'].unique())

print(f"Found {len(unique_rates)} Travel Rates and {len(unique_countries)} Seed Countries.")

for rate in unique_rates:
    for country in unique_countries:
        
        # Filter Data for this specific combination
        # 1. Filter Lines Data
        subset_lines = modelDat[
            (modelDat['param_travel_rate'] == rate) & 
            (modelDat['seed_country'] == country)
        ].copy()
        
        # 2. Filter Peaks Data
        subset_peaks = pd.DataFrame()
        if not peaks_summary.empty:
            subset_peaks = peaks_summary[
                (peaks_summary['param_travel_rate'] == rate) & 
                (peaks_summary['seed_country'] == country)
            ].copy()
        
        # Skip if no data exists for this combo
        if subset_lines.empty:
            continue

        print(f"\n=== Processing: Travel Rate {rate:.5f} | Seed Country {country} ===")
        
        # Melt line data for Seaborn
        melted_lines = subset_lines.melt(
            id_vars=['Day', 'Seed'], 
            value_vars=[f"{s}_InfectionPercent" for s in strategies],
            var_name='Strategy_Col', 
            value_name='Infection_Percent'
        )
        melted_lines['Strategy'] = melted_lines['Strategy_Col'].str.replace("_InfectionPercent", "")

        # Prepare Colors
        unique_strats = melted_lines['Strategy'].unique()
        palette = sns.color_palette('tab10', n_colors=len(unique_strats))
        strat_colors = dict(zip(unique_strats, palette))
        
        # Get unique seeds (iterations) for this specific combo
        current_seeds = subset_lines['Seed'].unique()
        
        # -----------------------------------------------------
        # GRAPHIC TYPE A: INDIVIDUAL SEED PLOTS (The 10 runs)
        # -----------------------------------------------------
        for seed in current_seeds:
            fig, axes = plt.subplots(1, 2, figsize=(18, 6))
            
            # Data for this specific seed
            this_seed_peaks = subset_peaks[subset_peaks['Seed'] == seed] if not subset_peaks.empty else pd.DataFrame()
            this_seed_lines = melted_lines[melted_lines['Seed'] == seed]
            
            # -- LEFT PLOT: BAR CHART (First Peak Day) --
            if not this_seed_peaks.empty:
                sns.barplot(
                    ax=axes[0],
                    data=this_seed_peaks,
                    x='Strategy',
                    y='Day',
                    hue='Strategy',
                    palette=strat_colors,
                    dodge=False,
                    legend=False,
                    errorbar=None
                )
                add_bar_labels(axes[0], fmt='%.0f')
                axes[0].set_title(f"First Peak Day")
                axes[0].tick_params(axis='x', rotation=45)
            else:
                axes[0].text(0.5, 0.5, "No Peak Data", ha='center', transform=axes[0].transAxes)
            
            # -- RIGHT PLOT: LINE CHART (Dynamics) --
            if not this_seed_lines.empty:
                sns.lineplot(
                    ax=axes[1],
                    data=this_seed_lines,
                    x='Day',
                    y='Infection_Percent',
                    hue='Strategy',
                    palette=strat_colors,
                    errorbar=None
                )
                
                # Add 'X' Markers and Labels for Peaks
                if not this_seed_peaks.empty:
                    for _, row in this_seed_peaks.iterrows():
                        day_val = row['Day']
                        inf_val = row['Peak_Infection_Rate']
                        axes[1].scatter(day_val, inf_val, color='black', marker='x', s=50, zorder=5)
                        axes[1].annotate(
                            f"({int(day_val)}, {inf_val:.1%})", 
                            xy=(day_val, inf_val), 
                            xytext=(5, 5), textcoords='offset points',
                            fontsize=8, color='black', fontweight='bold'
                        )

                axes[1].set_title(f"Infection Dynamics")
                axes[1].set_ylim(0, 1.0)
            
            # Header for the specific run
            plt.suptitle(f"Run Analysis: Travel Rate {rate:.5f} | Seed Country {country} | Seed {seed}", fontsize=14)
            plt.tight_layout()
            plt.show()

        # -----------------------------------------------------
        # GRAPHIC TYPE B: AVERAGE PLOT (Combined seeds for this Combo)
        # -----------------------------------------------------
        fig, axes = plt.subplots(1, 2, figsize=(18, 6))
        
        # -- LEFT PLOT: AVERAGE BAR CHART --
        if not subset_peaks.empty:
            sns.barplot(
                ax=axes[0],
                data=subset_peaks,
                x='Strategy',
                y='Day',
                hue='Strategy',
                palette=strat_colors,
                dodge=False,
                legend=False,
                errorbar=None
            )
            add_bar_labels(axes[0], fmt='%.1f')
            axes[0].set_title(f"AVERAGE Peak Day (Across {len(current_seeds)} Seeds)")
            axes[0].tick_params(axis='x', rotation=45)
        else:
            axes[0].text(0.5, 0.5, "No Peak Data", ha='center', transform=axes[0].transAxes)
            
        # -- RIGHT PLOT: AVERAGE LINE CHART --
        sns.lineplot(
            ax=axes[1],
            data=melted_lines,
            x='Day',
            y='Infection_Percent',
            hue='Strategy',
            palette=strat_colors,
            errorbar=None
        )
        
        # Add AVERAGE Peak Markers
        if not subset_peaks.empty:
            avg_peaks = subset_peaks.groupby('Strategy')[['Day', 'Peak_Infection_Rate']].mean().reset_index()
            for _, row in avg_peaks.iterrows():
                day_val = row['Day']
                inf_val = row['Peak_Infection_Rate']
                axes[1].scatter(day_val, inf_val, color='black', marker='x', s=50, zorder=5)
                axes[1].annotate(
                    f"({day_val:.1f}, {inf_val:.1%})", 
                    xy=(day_val, inf_val), 
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=8, color='black', fontweight='bold'
                )
        
        axes[1].set_title(f"AVERAGE Infection Dynamics (Across {len(current_seeds)} Seeds)")
        axes[1].set_ylim(0, 1.0)
        
        plt.suptitle(f"AGGREGATED SUMMARY: Travel Rate {rate:.5f} | Seed Country {country}", fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()